# Project 35 — LangGraph Resume Tailoring Flow
## Parse JD → Match Skills → Tailor Resume → Draft Cover Letter

**Stack:** LangGraph · Ollama · Jupyter

In [ ]:
# !pip install -q langgraph langchain langchain-ollama

## Step 1 — Setup

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.3)

class ResumeState(TypedDict):
    resume: str
    job_description: str
    jd_requirements: str
    skill_matches: str
    tailored_resume: str
    cover_letter: str

resume = """Software Engineer with 3 years experience. Built REST APIs with Python/FastAPI.
Led migration from monolith to microservices reducing deploy time by 60%.
Mentored 3 junior developers. Experience with PostgreSQL, Redis, Docker, AWS."""

jd = """Senior Backend Engineer — DataFlow Inc.
Requirements: 3+ years Python backend, distributed systems experience,
Kubernetes expertise, strong SQL skills, experience with data pipelines,
leadership and mentoring ability, excellent communication."""

## Step 2 — Define Pipeline Nodes

In [ ]:
def parse_jd(state: ResumeState) -> ResumeState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Extract a structured list of requirements from this JD. "
         "Categorize as: must-have, nice-to-have."),
        ("human", "{jd}")
    ])
    chain = prompt | llm | StrOutputParser()
    reqs = chain.invoke({"jd": state["job_description"]})
    return {"jd_requirements": reqs}

def match_skills(state: ResumeState) -> ResumeState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Compare the resume against JD requirements. List: "
         "MATCHED skills, PARTIALLY MATCHED, and GAPS."),
        ("human", "Resume: {resume}\n\nRequirements: {reqs}")
    ])
    chain = prompt | llm | StrOutputParser()
    matches = chain.invoke({"resume": state["resume"], "reqs": state["jd_requirements"]})
    return {"skill_matches": matches}

def tailor_resume(state: ResumeState) -> ResumeState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Rewrite the resume to better match the JD. "
         "Emphasize matching skills, use JD keywords, add metrics. "
         "Do NOT fabricate experience."),
        ("human", "Resume: {resume}\nMatches: {matches}\nJD: {jd}")
    ])
    chain = prompt | llm | StrOutputParser()
    tailored = chain.invoke({
        "resume": state["resume"], "matches": state["skill_matches"],
        "jd": state["job_description"],
    })
    return {"tailored_resume": tailored}

def write_cover_letter(state: ResumeState) -> ResumeState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Write a cover letter connecting resume strengths to JD needs. "
         "Under 200 words."),
        ("human", "Tailored Resume: {resume}\nJD: {jd}\nGaps: {matches}")
    ])
    chain = prompt | llm | StrOutputParser()
    letter = chain.invoke({
        "resume": state["tailored_resume"], "jd": state["job_description"],
        "matches": state["skill_matches"],
    })
    return {"cover_letter": letter}

## Step 3 — Build and Run

In [ ]:
graph = StateGraph(ResumeState)
graph.add_node("parse_jd", parse_jd)
graph.add_node("match_skills", match_skills)
graph.add_node("tailor_resume", tailor_resume)
graph.add_node("write_cover_letter", write_cover_letter)

graph.set_entry_point("parse_jd")
graph.add_edge("parse_jd", "match_skills")
graph.add_edge("match_skills", "tailor_resume")
graph.add_edge("tailor_resume", "write_cover_letter")
graph.add_edge("write_cover_letter", END)

app = graph.compile()

result = app.invoke({
    "resume": resume, "job_description": jd,
    "jd_requirements": "", "skill_matches": "",
    "tailored_resume": "", "cover_letter": "",
})

print("=== SKILL ANALYSIS ===")
print(result["skill_matches"])
print("\n=== TAILORED RESUME ===")
print(result["tailored_resume"])
print("\n=== COVER LETTER ===")
print(result["cover_letter"])

## What You Learned
- **JD parsing** into structured requirements
- **Skill gap analysis** comparing resume vs JD
- **Automated resume tailoring** preserving authenticity
- **Multi-step document generation** pipeline